In [3]:
import sys
import os

sys.path.append(os.path.abspath(".."))

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from load_data import load_images, load_labels

import tensorflow as tf
from tensorflow import keras

In [4]:
model = tf.keras.models.load_model("mnist_model_modified_data.keras")

#Load test data 
# Load test data
X_test = load_images("../MNIST_dataset/t10k-images-idx3-ubyte/t10k-images-idx3-ubyte")
y_test = load_labels("../MNIST_dataset/t10k-labels-idx1-ubyte/t10k-labels-idx1-ubyte")

# Preprocess test data
X_test = X_test.reshape(X_test.shape[0], -1) / 255.0
y_test_encoded = keras.utils.to_categorical(y_test, num_classes=10)
test_loss, test_accuracy = model.evaluate(X_test, y_test_encoded, verbose=0)
print(f"Final Test Accuracy on unseen test set: {test_accuracy:.4f}")

Final Test Accuracy on unseen test set: 0.9706


In [5]:
# !pip install ipycanvas

In [ ]:
from ipycanvas import Canvas
from ipywidgets import Layout
from ipywidgets import Button, VBox
from ipywidgets import Output
from IPython.display import display

canvas_size = 28
canvas = Canvas(
    width=canvas_size,
    height=canvas_size,
    sync_image_data=True,
    layout=Layout(width=f'300px', height=f'300px')
)

canvas.fill_style = "black"
canvas.fill_rect(0,0,canvas_size,canvas_size)

canvas.stroke_style = "white"
canvas.line_width = 1

drawing = False

def on_mouse_down(x, y):
    global drawing
    drawing = True
    canvas.begin_path()
    canvas.move_to(x,y)

def on_mouse_move(x,y):
    if drawing:
        canvas.line_to(x,y)
        canvas.stroke()

def on_mouse_up(x,y):
    global drawing
    drawing = False

canvas.on_mouse_down(on_mouse_down)
canvas.on_mouse_move(on_mouse_move)
canvas.on_mouse_up(on_mouse_up)

def get_image():
    img = np.array(canvas.get_image_data(0,0,canvas_size,canvas_size))

    img = img[:,:,0]      # grayscale
    img = img / 255.0

    return img.reshape(1,784)


predict_button = Button(description="Predict")

out = Output()

def predict_digit(b):
    with out:
        out.clear_output()
        img = get_image()
        pred = model.predict(img)
        print("Prediction:", np.argmax(pred))

predict_button.on_click(predict_digit)


clear_button = Button(description="Clear")

def clear_canvas(b):
    canvas.fill_rect(0,0,canvas_size,canvas_size)

clear_button.on_click(clear_canvas)

from IPython.display import HTML, display
display(HTML("""
<style>
canvas {
    image-rendering: pixelated !important;s
}
</style>
"""))

display(VBox([canvas, predict_button, clear_button, out]))